# Fine-tuning d'un LLM sur TeleQnA (normes 3GPP)
## Utilisation : Kaggle (GPU T4 x2, 30h/semaine gratuit)

### Avant de commencer
1. Va sur kaggle.com > Create New Notebook
2. Settings > Accelerator > **GPU T4 x2**
3. Settings > Internet > **ON**
4. Ajoute le fichier `telecom_train.json` en tant que Dataset (Upload ou lien Kaggle)
5. Copie-colle ce notebook

In [ ]:
# ============================================================
# 1. Installer Unsloth (optimise le fine-tuning sur GPU)
# ============================================================
import torch
print(f"GPU : {torch.cuda.get_device_name()}" if torch.cuda.is_available() else "CPU")

!pip install unsloth -q
!pip install --force-reinstall --no-cache-dir unsloth -q

In [ ]:
# ============================================================
# 2. Charger les donnees TeleQnA
# ============================================================
import json, re
from pathlib import Path

# Chercher le fichier (Kaggle le met dans /kaggle/input/...)
files = list(Path('/kaggle/input').rglob('*.json'))
print('Fichiers trouves :', [f.name for f in files])

if files:
    with open(files[0]) as f:
        data = json.load(f)
else:
    # Fallback : upload direct
    from google.colab import files
    uploaded = files.upload()
    with open(list(uploaded.keys())[0]) as f:
        data = json.load(f)

print(f"Charge : {len(data)} questions TeleQnA")

In [ ]:
# ============================================================
# 3. Formater les donnees pour le fine-tuning
# ============================================================
# Format : instruction QCM avec la bonne reponse

def format_exemple(item):
    opts = "\n".join(f"{chr(65+i)}) {o}" for i, o in enumerate(item["options"]))
    reponse_letter = chr(65 + int(re.search(r'option (\\d+)', item["reponse"]).group(1)) - 1) if re.search(r'option (\\d+)', item["reponse"]) else "A"
    explication = item.get("explication", "").strip()
    
    prompt = f"""### Contexte :
Vous etes un expert en telecommunications 3GPP.
Repondez a la question QCM en donnant la lettre de la bonne reponse et une explication.

### Question :
{item['question']}

### Options :
{opts}

### Reponse :
{reponse_letter}
"""
    
    if explication:
        prompt += f"\n### Explication :\n{explication}"
    
    return {"text": prompt}

formatted = [format_exemple(item) for item in data]
print(f"Formate : {len(formatted)} exemples")
print("\nExemple :")
print(formatted[0]["text"][:500])

In [ ]:
# ============================================================
# 4. Charger le modele de base + tokenizer
# ============================================================
# Choix du modele :
#   - "unsloth/Llama-3.1-8B-Instruct-bnb-4bit"  (8B, meilleur qualite)
#   - "unsloth/Mistral-7B-Instruct-v0.3-bnb-4bit" (7B, rapide)
#   - "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  (7B, multilingue)
#   - "unsloth/Phi-3-mini-4k-instruct"  (3.8B, leger)

MODEL_NAME = "unsloth/Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # Auto-detect
    load_in_4bit=True,   # QLoRA : 4 bits
)

print(f"Modele charge : {MODEL_NAME}")
print(f"Taille : {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B parametres")

In [ ]:
# ============================================================
# 5. Configurer LoRA
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,              # Rang LoRA (16 = bon compromis)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,     # Scaling
    lora_dropout=0,    # 0 = optimise
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA configure :", sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, "M parametres entrainables")

In [ ]:
# ============================================================
# 6. Lancer l'entrainement
# ============================================================
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",  # Pas de wandb
    ),
)

print("Entrainement...")
trainer.train()

In [ ]:
# ============================================================
# 7. Sauvegarder le modele
# ============================================================
# Adapter LoRA seul (~16 Mo)
model.save_pretrained("telecom_lora_adapter")
tokenizer.save_pretrained("telecom_lora_adapter")

# Modele complet en 16 bits (~16 Go pour un 8B)
# model.save_pretrained_merged("telecom_model_16bit", tokenizer, save_method="merged_16bit")

# OU export GGUF pour llama.cpp / Ollama (~5 Go)
# model.save_pretrained_gguf("telecom_model_gguf", tokenizer, quantization_method="q4_k_m")

print("Modele sauvegarde !")
print("\nPour Kaggle : telecharge le dossier via Kaggle > Data > Output")

In [ ]:
# ============================================================
# 8. Tester le modele fine-tune
# ============================================================
FastLanguageModel.for_inference(model)

test_question = "What is the role of the AMF in 5G core network?"
prompt = f"""### Contexte :
Vous etes un expert en telecommunications 3GPP.
Repondez a la question QCM en donnant la lettre de la bonne reponse et une explication.

### Question :
{test_question}

### Reponse :
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Reponse du modele fine-tune :")
print(response[len(prompt):])

## Prochaines etapes

1. **Telecharger** le dossier `telecom_lora_adapter` depuis Kaggle > Output
2. **Utiliser le modele localement** via Ollama ou llama.cpp :
   - Convertir en GGUF avec `model.save_pretrained_gguf(...)`
   - Lancer avec `llama-server`
3. **Integrer dans le projet RAG** : remplacer `LLM_MODEL` dans le `.env`

### Pour aller plus loin
- Augmenter `num_train_epochs=3` pour plus de repetitions
- Ajouter les donnees de test (`telecom_test.json`) pour evaluer
- Essayez `Qwen2.5-7B` qui supporte mieux le multilangue